[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_6_n_step_sarsa.ipynb)

<div style="text-align:center">
    <h1>
        n-step SARSA
    </h1>
</div>

<br><br>

<div style="text-align:center">
    In this notebook we are going to combine the temporal difference method SARSA with n-step bootstrapping. The resulting algorithm is called n-step SARSA and uses the following target for the updates:
</div>

\begin{equation}
\hat G_t = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{n} + \gamma Q(S_n, A_n)
\end{equation}

<br>

<div style="text-align:center">
    This method follows an on-policy strategy, in which the same policy that is optimized is responsible for exploring the environment.
</div>



In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_agent, plot_policy, plot_action_values


## Import the necessary software libraries:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Create the environment, value table and policy

#### Create the environment

In [ ]:
env = Maze()

#### Create the $Q(s, a)$ table

In [ ]:
action_values = np.zeros(shape=(5, 5, 4))

#### Create the policy $\pi(s)$

In [ ]:
def policy(state, epsilon=0.):
    if np.random.random() < epsilon:
        return np.random.randint(4)
    else:
        av = action_values[state]
        return np.random.choice(np.flatnonzero(av == av.max()))

#### Plot the value table $Q(s,a)$

In [ ]:
plot_action_values(action_values)

#### Plot the policy

In [ ]:
plot_policy(action_values, env.render(mode='rgb_array'))

## Implement the algorithm


### What you are building: n-step SARSA

Ordinary SARSA learns from just **one** real step of reward plus a guess of the rest.
Early on that guess is poor, so learning is slow. **n-step SARSA** takes $n$ **real**
steps first, adds up those (discounted) rewards, and only then bootstraps with a guess
of what comes after step $n$. More real reward and less guessing usually means
**faster, steadier learning**.

**The n-step return (the target you will build):**

$$\hat G_t = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{\,n-1} R_{t+n} + \gamma^{\,n} Q(S_{t+n}, A_{t+n})$$

The first part is a chain of **real** discounted rewards; the last term is the single
**bootstrapped** estimate at the end of the window. The update is then:

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha\,\bigl[\hat G_t - Q(S_t, A_t)\bigr]$$

**$n$ is a dial:**
- $n = 1$ is ordinary one-step SARSA (most bootstrapping).
- very large $n$ behaves like Monte Carlo (almost no bootstrapping).
- medium values (e.g. $n = 8$) often learn fastest.

n-step SARSA stays **on-policy** - it uses the next action the policy actually chooses.

**Your tasks below (fill in the blank cells):**
1. Write the `n_step_sarsa(...)` loop. The tricky part: you cannot compute $\hat G_t$
   until $n$ later steps have happened, so keep a small **buffer of recent
   transitions** and, at each time step, update the state-action pair from $n$ steps
   ago.
2. Remember to keep looping a little past the end of an episode so the final few
   updates still get applied.
3. Run it, then plot and test the learned policy.

*Hint: store `[state, action, reward]` each step; once you have n of them, sum the last
n rewards backwards with discounting and add a bootstrapped `Q` value at the end.*

## Show results

#### Show resulting value table $Q(s, a)$

In [ ]:
plot_action_values(action_values)

#### Show resulting policy $\pi(\cdot|s)$

In [ ]:
plot_policy(action_values, env.render(mode='rgb_array'))

#### Test the resulting agent

In [ ]:
test_agent(env, policy)

## Resources

[[1] Reinforcement Learning: An Introduction. Ch. 7: n-step bootstrapping](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf)